# 3 - Full pipeline

> 📗 **About this browser version:** this notebook runs in JupyterLite, in your browser, so there is nothing to set up. The `data/` folder holds a sample of 200 of the 1,007 invoice PDFs in the original repository, which keeps the loops over every file quick in the browser. The sample includes a few invoices that the patterns do not match, as the full set does. The first code cell installs pdfplumber 0.9.0: newer versions need a library (`pypdfium2`) that cannot run in the browser, and the parts of pdfplumber these notebooks use work the same way in 0.9.0.

In [ ]:
# pdfplumber 0.9.0 runs in the browser; newer versions need pypdfium2, which does not
%pip install -q pdfplumber==0.9.0

import os
import sqlite3
import pdfplumber
import json
import re

In [ ]:
# Directory containing all PDF files
pdf_directory = "data/"
pdf_files = [os.path.join(pdf_directory, f) for f in os.listdir(pdf_directory) if f.endswith(".pdf")]

In [ ]:
orderid_pattern = r'[A-Z]{2}-\d{4}-[A-Z]{2}\d{5,}-\d{5}'
shipmode_pattern = r"Ship Mode: ([A-Za-z\s]+)(?=\n)"
date_pattern = r"([A-Za-z]+\s\d{1,2}\s\d{4})"
itemcode_pattern = r"\b[A-Z]{3}-[A-Z]{2}-\d{4}\b"
subtotal_pattern = r"Subtotal: \$(\d[\d,]*\.\d{2})"
discount_pattern = r"Discount.*?: \$(\d[\d,]*\.\d{2})"
shipping_pattern = r"Shipping: \$(\d[\d,]*\.\d{2})"
total_pattern = r"Total: \$(\d[\d,]*\.\d{2})"

In [ ]:
order_data = []
for pdf in pdf_files:
    with pdfplumber.open(pdf) as pdf:
        for page in pdf.pages:
            text = page.extract_text()                                
            
            orderid = re.search(orderid_pattern, text).group() if re.search(orderid_pattern, text) else None
            date = re.search(date_pattern, text).group() if re.search(date_pattern, text) else None
            itemcode = re.search(itemcode_pattern, text).group() if re.search(itemcode_pattern, text) else None
            shipmode = re.search(shipmode_pattern, text).group(1) if re.search(shipmode_pattern, text) else None
            subtotal = re.search(subtotal_pattern, text).group(1) if re.search(subtotal_pattern, text) else None
            discount = re.search(discount_pattern, text).group(1) if re.search(discount_pattern, text) else None
            shipping = re.search(shipping_pattern, text).group(1) if re.search(shipping_pattern, text) else None
            total = re.search(total_pattern, text).group(1) if re.search(total_pattern, text) else None
            
            if orderid:
                # Create a JSON object for the rest of the data
                data = {
                    "date": date,
                    "item_code": itemcode,
                    "ship_mode": shipmode,
                    "subtotal": float(subtotal.replace(",", "")) if subtotal else 0.0,
                    "discount": float(discount.replace(",", "") if discount else 0.0),
                    "shipping": float(shipping.replace(",", "") if shipping else 0.0),
                    "total": float(total.replace(",", "") if total else 0.0)
                }
                order_data.append((orderid, json.dumps(data)))
            

In [ ]:
order_data[0:5]

In [ ]:
# SQLite database file
db_name = "testing.db"
# In the browser, SQLite cannot write a database straight into the file browser,
# so we build it in memory and save it to testing.db once the orders are in (see below)
conn = sqlite3.connect(":memory:")

In [ ]:
with conn:
    conn.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        order_id TEXT NOT NULL,
        order_details TEXT NOT NULL
    )
    """)
    
    conn.executemany("""
    INSERT INTO orders (order_id, order_details)
    VALUES (?, ?)
    """, order_data)

Save the database to the file `testing.db`. (Run on your own machine, `sqlite3.connect(db_name)` writes to the file directly, so this step is not needed there.)

In [ ]:
with open(db_name, "wb") as f:
    f.write(conn.serialize())

Check what was written: count the rows in the `orders` table and look at the first few. The database file `testing.db` appears in the file browser on the left. Each run of the insert cell adds the orders again, so the count grows if you run it more than once.

In [ ]:
for row in conn.execute("SELECT COUNT(*) FROM orders"):
    print(row)
for row in conn.execute("SELECT * FROM orders LIMIT 5"):
    print(row)